# SFT → merge → DPO → merge (Kaggle-friendly, from scratch)

**Model:** `Qwen/Qwen2.5-0.5B` (base) | **GPU:** Kaggle T4 / P100 | **Internet:** ON

```
BASE model
   │  Step 1: LoRA-A  (instruction SFT)
   ▼
merge LoRA-A into weights  ──►  sft-merged   (a plain model, no adapter)
   │  Step 2: NEW LoRA-B  (DPO)
   ▼
merge LoRA-B into weights  ──►  dpo-merged   (a plain model, no adapter)
```

> ⚠️ **Rule of this notebook:** never stack LoRA on LoRA. After each stage we **merge** the adapter, and the next stage starts from a clean, plain model.

In [ ]:
!pip install -q -U trl peft datasets

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU only (avoids DataParallel surprises on T4 x2)
os.environ["WANDB_DISABLED"] = "true"

import gc, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer

BASE  = "Qwen/Qwen2.5-0.5B"
BF16  = torch.cuda.is_bf16_supported()          # T4/P100 -> False -> we use fp16
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", BF16)

def chat(model, tok, prompt, max_new_tokens=150):
    text = tok.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

def free():
    gc.collect(); torch.cuda.empty_cache()

---
# STEP 1 — Instruction fine-tuning (SFT)
Minimal: base model + chat data + LoRA → merge → save.

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
tok.eos_token = "<|im_end|>"        # end-of-turn token used by the chat template
tok.pad_token = "<|endoftext|>"

model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=DTYPE).to("cuda")

sft_data = load_dataset("trl-lib/Capybara", split="train").select(range(2000))   # 'messages' format
print(sft_data[0]["messages"][:2])

In [ ]:
sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_data,
    processing_class=tok,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                           target_modules="all-linear", task_type="CAUSAL_LM"),   # LoRA-A
    args=SFTConfig(
        output_dir="sft-out", num_train_epochs=1,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
        max_length=512, logging_steps=10, save_strategy="no",
        bf16=BF16, fp16=not BF16, report_to="none",
    ),
)
sft_trainer.train()

In [ ]:
# MERGE LoRA-A into the base weights -> plain model (no adapter left)
sft_model = sft_trainer.model.merge_and_unload()
sft_model.generation_config.eos_token_id = tok.eos_token_id
sft_model.save_pretrained("sft-merged")
tok.save_pretrained("sft-merged")

print(chat(sft_model, tok, "Give me three tips for learning Python."))

del sft_trainer, sft_model, model; free()

---
# STEP 2 — DPO with LoRA, then merge

### Why we merge instead of stacking
- After Step 1 we saved **`sft-merged`**: a normal model whose weights already contain the SFT knowledge. There is **no adapter** on it.
- In Step 2 we load `sft-merged` as if it were the base model and attach a **brand-new LoRA-B**.
- We do **not** load LoRA-A again. Stacking adapters (LoRA on LoRA) makes the reference model ambiguous, complicates saving/serving, and is easy to get wrong.

### What DPO uses as the reference model here
With `ref_model=None` and a `peft_config`, TRL computes the reference log-probs by **disabling the adapter** on the same model. Since the underlying weights are `sft-merged`, the reference is exactly **the SFT model**. That is what DPO wants.

```
policy    = sft-merged + LoRA-B   (trained)
reference = sft-merged            (adapter switched off, frozen)
```

In [ ]:
tok = AutoTokenizer.from_pretrained("sft-merged")
model = AutoModelForCausalLM.from_pretrained("sft-merged", torch_dtype=DTYPE).to("cuda")   # plain model, no adapter

# preference pairs: prompt / chosen / rejected
dpo_data = load_dataset("trl-lib/ultrafeedback_binarized", split="train").select(range(2000))
print(dpo_data.column_names)

In [ ]:
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,                     # reference = same model with adapter disabled (= SFT model)
    train_dataset=dpo_data,
    processing_class=tok,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                           target_modules="all-linear", task_type="CAUSAL_LM"),   # NEW LoRA-B
    args=DPOConfig(
        output_dir="dpo-out", num_train_epochs=1, beta=0.1,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-5, lr_scheduler_type="cosine", warmup_ratio=0.05,
        max_length=768, logging_steps=10, save_strategy="no",
        bf16=BF16, fp16=not BF16, report_to="none",
    ),
)
dpo_trainer.train()

In [ ]:
# MERGE LoRA-B into sft-merged -> final plain model
dpo_model = dpo_trainer.model.merge_and_unload()
dpo_model.generation_config.eos_token_id = tok.eos_token_id
dpo_model.save_pretrained("dpo-merged")
tok.save_pretrained("dpo-merged")
print("saved -> dpo-merged")

### Sanity checks: no adapters left, and weights really changed

In [ ]:
print("class:", type(dpo_model).__name__)                                   # Qwen2ForCausalLM, not PeftModel
print("LoRA params left:", sum("lora" in n for n, _ in dpo_model.named_parameters()))   # must be 0

ref = AutoModelForCausalLM.from_pretrained("sft-merged", torch_dtype=DTYPE)
k = "model.layers.0.self_attn.q_proj.weight"
diff = (dpo_model.state_dict()[k].cpu().float() - ref.state_dict()[k].float()).abs().max().item()
print(f"max |dpo-merged - sft-merged| on {k}: {diff:.6f}   (> 0 means DPO changed the weights)")
del ref; free()

print("\n--- dpo-merged answer ---")
print(chat(dpo_model, tok, "Give me three tips for learning Python."))

Outputs are in `/kaggle/working/`: `sft-merged/` and `dpo-merged/`. Both are standard Hugging Face models, so load them with `AutoModelForCausalLM.from_pretrained(...)`, no PEFT needed.